In [ ]:
import csv
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import simplekml
import random
from collections import defaultdict

file_path = "trajectory.csv"

In [ ]:
flights = defaultdict(list)

with open(file_path, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        fs = int(row["firstseen"])
        t = int(row["time"])
        lat = float(row["lat"])
        lon = float(row["lon"])
        flights[fs].append((t, lat, lon))

# sort each flight by time
for fs in flights:
    flights[fs] = sorted(flights[fs], key=lambda x: x[0])

all_lats = []
all_lons = []

for flight in flights.values():
    for _, lat, lon in flight:
        all_lats.append(lat)
        all_lons.append(lon)

all_lats = np.array(all_lats)
all_lons = np.array(all_lons)

lat_mean = np.mean(all_lats)
lat_std = np.std(all_lats)
lon_mean = np.mean(all_lons)
lon_std = np.std(all_lons)

In [ ]:
SEQ_LEN = 100

def resample_path(path, target_len):
    idx = np.linspace(0, len(path) - 1, target_len).astype(int)
    return path[idx]

X = []
y = []

for flight in flights.values():
    if len(flight) < 10:
        continue
    
    coords = np.array([[lat, lon] for _, lat, lon in flight])
    
    coords_norm = np.stack([
        (coords[:,0] - lat_mean) / lat_std,
        (coords[:,1] - lon_mean) / lon_std
    ], axis=1)

    path = resample_path(coords_norm, SEQ_LEN)

    start = path[0]
    end = path[-1]

    X.append([start[0], start[1], end[0], end[1]])
    y.append(path)

X = np.array(X)
y = np.array(y)

In [ ]:
def build_model(seq_len):
    inputs = layers.Input(shape=(4,))  # start + end

    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(seq_len * 2)(x)

    outputs = layers.Reshape((seq_len, 2))(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")

    return model

In [ ]:
model = build_model(SEQ_LEN)
model.fit(X, y, epochs=200, verbose=1)

In [ ]:
sample_idx = 0  # choose which flight to test

X_test = X[sample_idx:sample_idx+1]
real_path = y[sample_idx]

predicted = model.predict(X_test)[0]

In [ ]:
def denorm(path):
    lat = path[:,0] * lat_std + lat_mean
    lon = path[:,1] * lon_std + lon_mean
    return np.stack([lat, lon], axis=1)

real_path = denorm(real_path)
predicted_real = denorm(predicted)

In [ ]:
kml = simplekml.Kml()

for fs, flight in flights.items():
    coords = [(lon, lat) for _, lat, lon in flight]
    
    line = kml.newlinestring(
        name=f"Real Flight {fs}",
        coords=coords
    )
    line.style.linestyle.color = simplekml.Color.green

predicted_coords = [(lon, lat) for lat, lon in predicted_real]
line_pred = kml.newlinestring(
    name="Predicted Flight",
    coords=predicted_coords
)
line_pred.style.linestyle.color = simplekml.Color.red

kml.save("flight_prediction.kml")